Для інсталювання необхідного пакету в середовищі Anaconda потрібно ввести в терміналі Anaconda:

```python
pip install pulp
pip install numpy
```

Для інсталювання необхідного пакету в середовищі Google Colab потрібно ввести першій комірці:

```python
!pip install pulp
!pip install numpy
```

Більше деталей за [посиланням](https://colab.research.google.com/notebooks/snippets/importing_libraries.ipynb).

## Дано

Нехай маємо математичну модель траспортної задачі лінійного програмування (ТЗЛП):

$$\begin{array}{rcl} f = \min\limits_{x \in X} \{ 3x_{11} + 4x_{12} + 6x_{13} + 7x_{14} + \\
+ 2x_{21} + 5x_{22} + 8x_{23} + x_{24} \}, \\
\end{array}$$

$$\begin{cases}
\begin{aligned}
x_{11} + x_{12} + x_{13} + x_{14} = 60; \\
x_{21} + x_{22} + x_{23} + x_{24} = 40, \\
\end{aligned}
\end{cases}$$

$$\begin{cases}
\begin{aligned}
x_{11} + x_{21} = 20; \\
x_{12} + x_{22} = 30; \\
x_{13} + x_{23} = 45; \\
x_{14} + x_{24} = 15, \\
\end{aligned}
\end{cases}$$

$$\begin{array}
1x_{ij} \geq 0, \quad i = \overline{1,2}, \quad j = \overline{1,4}.
\end{array}$$

## Блок 1. Ініціалізація вхідних даних

In [65]:
from pulp import *
import numpy as np

In [66]:
# Склади та торгівельні точки: List
Warehouse = ['A1', 'A2']
Branch = ['B1', 'B2', 'B3', 'B4']

In [67]:
# Об'єми вивозу
demand = {'A1' : 60, 'A2' : 40}

In [68]:
# Об'єми поставок
supply = {'B1' : 20, 'B2' : 30, 'B3' : 45, 'B4' : 15}

In [69]:
# Витрати на перевезення одиниці вантажу із складів до торгівельних точок
cost = {'A1': {'B1' : 3, 'B2' : 4, 'B3' : 6, 'B4' : 7},
        'A2': {'B1' : 2, 'B2' : 5, 'B3' : 8, 'B4' : 1},
       }

## Блок 2. Реалізація транспортної задачі лінійного програмування

In [70]:
# Створення об'єкту математичної моделі
probTrans = LpProblem("ТЗЛП", LpMinimize)

In [71]:
# Оголошення матриці, що моделює всі можливі шляхи перевезення 
routes = [(i,j) for i in Warehouse for j in Branch]
print(routes)

[('A1', 'B1'), ('A1', 'B2'), ('A1', 'B3'), ('A1', 'B4'), ('A2', 'B1'), ('A2', 'B2'), ('A2', 'B3'), ('A2', 'B4')]


In [72]:
# Оголошення змінних мат. моделі
amount_vars = LpVariable.dicts(name="X", indexs=(Warehouse,Branch), lowBound=0, upBound=None)
print(amount_vars)

{'A1': {'B1': X_A1_B1, 'B2': X_A1_B2, 'B3': X_A1_B3, 'B4': X_A1_B4}, 'A2': {'B1': X_A2_B1, 'B2': X_A2_B2, 'B3': X_A2_B3, 'B4': X_A2_B4}}


C:\Users\radiu\AppData\Local\Temp/ipykernel_18136/3197697366.py:2: DeprecationWarning: 'indexs' is deprecated; use 'indices'.
  amount_vars = LpVariable.dicts(name="X", indexs=(Warehouse,Branch), lowBound=0, upBound=None)


In [73]:
# Оголошення та ініціалізація цільової функції
# lpSum - обчислення суми добутків C*X
probTrans += lpSum(amount_vars[i][j]*cost[i][j] for (i,j) in routes)

In [74]:
# Оголошення та ініціалізація обмежень моделі
for i in Warehouse:
    probTrans += lpSum(amount_vars[i][j] for j in Branch) == demand[i]

for j in Branch:
    probTrans += lpSum(amount_vars[i][j] for i in Warehouse) <= supply[j]

In [75]:
# Відображення математичної моделі
print(probTrans)

ТЗЛП:
MINIMIZE
3*X_A1_B1 + 4*X_A1_B2 + 6*X_A1_B3 + 7*X_A1_B4 + 2*X_A2_B1 + 5*X_A2_B2 + 8*X_A2_B3 + 1*X_A2_B4 + 0
SUBJECT TO
_C1: X_A1_B1 + X_A1_B2 + X_A1_B3 + X_A1_B4 = 60

_C2: X_A2_B1 + X_A2_B2 + X_A2_B3 + X_A2_B4 = 40

_C3: X_A1_B1 + X_A2_B1 <= 20

_C4: X_A1_B2 + X_A2_B2 <= 30

_C5: X_A1_B3 + X_A2_B3 <= 45

_C6: X_A1_B4 + X_A2_B4 <= 15

VARIABLES
X_A1_B1 Continuous
X_A1_B2 Continuous
X_A1_B3 Continuous
X_A1_B4 Continuous
X_A2_B1 Continuous
X_A2_B2 Continuous
X_A2_B3 Continuous
X_A2_B4 Continuous



In [76]:
# Розв'язок ТЗЛП
probTrans.solve(PULP_CBC_CMD(msg=1))

1

## Блок 3. Виведення результату розв'язку транспортної задачі лінійного програмування

In [77]:
# Виведення результату оптимізації
print("Status:", LpStatus[probTrans.status])

Status: Optimal


In [78]:
# Виведення значень перевезень
for v in probTrans.variables():
    if v.varValue > 0:
        print(v.name, "=", v.varValue)

X_A1_B2 = 25.0
X_A1_B3 = 35.0
X_A2_B1 = 20.0
X_A2_B2 = 5.0
X_A2_B4 = 15.0


In [79]:
print("Оптимальне значення цільової функції: f_min =", value(probTrans.objective))

Оптимальне значення цільової функції: f_min = 390.0


In [80]:
outputPlan = []

for v in probTrans.variables():
    outputPlan.append(int(v.varValue))

outputPlan = np.array(outputPlan)
outputPlan = outputPlan.reshape(len(Warehouse),len(Branch))
print("Матричне виведення результату:")
print(outputPlan)

Матричне виведення результату:
[[ 0 25 35  0]
 [20  5  0 15]]
